# 🤗 Fine-Tuning BERT for Sentiment Analysis

**Institution**: Innomatics Research Labs — NLP Assignment 4

### 📝 Project Overview
This project demonstrates how to fine-tune a pre-trained **BERT (Bidirectional Encoder Representations from Transformers)** model for text classification. We specifically use the **IMDB Movie Reviews** dataset to perform sentiment analysis, comparing a feature-extraction approach (frozen layers) against partial fine-tuning (unfreezing specific layers).

### 🚀 Pipeline
1.  **Environment Setup**: Install and import required libraries.
2.  **Data Acquisition**: Load the IMDB dataset from HuggingFace.
3.  **Preprocessing**: Clean HTML tags, special characters, and normalize text.
4.  **Tokenization**: Convert text into BERT-compatible input IDs and attention masks.
5.  **Architecture**: Initialize `bert-base-uncased` with a classification head.
6.  **Comparative Experiments**: Execute training for 'Frozen' vs 'Unfrozen' layers.
7.  **Evaluation**: Analyze performance using Accuracy, F1-Score, and Confusion Matrices.

## 🧪 Experimental Design
We perform two distinct experiments to understand the impact of fine-tuning:

| Experiment | Strategy | Description |
| :--- | :--- | :--- |
| **Exp 1: Feature Extraction** | All BERT Layers Frozen | Only the newly added Classifier Head is trained. |
| **Exp 2: Partial Fine-Tuning** | Last 2 Layers Unfrozen | The last two encoder blocks and the classifier head are trained. |

## ⚙️ Step 1 — Install Dependencies

In [ ]:
!pip install -q transformers datasets scikit-learn seaborn tqdm

## 📦 Step 2 — Imports

In [ ]:
import re
import warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# Configure environment
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on device: {device}")

## 📊 Step 3 — Load Dataset

In [ ]:
print("Loading IMDB dataset...")
dataset = load_dataset("imdb")

train_df = pd.DataFrame(dataset['train'])
test_df  = pd.DataFrame(dataset['test'])

print(f"Training set: {train_df.shape}")
print(f"Testing set: {test_df.shape}")

## 🧹 Step 4 — Data Preprocessing

In [ ]:
def clean_text(text):
    """Removes HTML tags, punctuation, and normalizes spacing."""
    text = re.sub(r'<[^>]+>', ' ', text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9.,!?'\s]", ' ', text) # Remove special characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove redundant spaces
    return text.lower()

# Apply efficient cleaning
tqdm.pandas(desc="Cleaning Text")
train_df['text'] = train_df['text'].progress_apply(clean_text)
test_df['text']  = test_df['text'].progress_apply(clean_text)

## ✂️ Step 5 — Data Splitting

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    train_df['text'], train_df['label'],
    test_size=0.2, random_state=42
)

X_test = test_df['text']
y_test = test_df['label']

## 🔤 Step 6 — Tokenization

In [ ]:
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(texts):
    return tokenizer(
        list(texts),
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    )

## 📦 Step 7 — Dataset Class

In [ ]:
class IMDBDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_function(texts)
        self.labels = torch.tensor(labels.values)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

## 🤖 Step 8 — Model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
).to(device)

## ⚡ Step 9 — Training Setup

In [ ]:
optimizer = AdamW(model.parameters(), lr=2e-5)

train_dataset = IMDBDataset(X_train, y_train)
val_dataset   = IMDBDataset(X_val, y_val)
test_dataset  = IMDBDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)

## 🔁 Step 10 — Training Loop

In [ ]:
class BERTManager:
    """Handles training, evaluation, and experiment orchestration with Mixed Precision."""
    def __init__(self, model_name, device, num_labels=2):
        self.model_name = model_name
        self.device = device
        self.num_labels = num_labels
        self.scaler = torch.cuda.amp.GradScaler() # For Mixed Precision (FP16)

    def get_model(self):
        return AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels
        ).to(self.device)

    def train_epoch(self, model, loader, optimizer, scheduler):
        model.train()
        total_loss = 0
        loop = tqdm(loader, leave=False, desc="Training")

        for batch in loop:
            optimizer.zero_grad()
            inputs = {k: v.to(self.device) for k, v in batch.items()}

            # Mixed Precision Forward Pass
            with torch.cuda.amp.autocast():
                outputs = model(**inputs)
                loss = outputs.loss

            # Scaled Backward Pass
            self.scaler.scale(loss).backward()
            self.scaler.step(optimizer)
            self.scaler.update()
            scheduler.step()

            total_loss += loss.item()
            loop.set_description(f"Loss: {loss.item():.4f}")
        return total_loss / len(loader)

    def validate(self, model, loader):
        model.eval()
        all_preds, all_labels = [], []

        with torch.no_grad():
            for batch in tqdm(loader, desc="Validating", leave=False):
                inputs = {k: v.to(self.device) for k, v in batch.items()}
                outputs = model(**inputs)

                preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(batch['labels'].numpy())
        return all_labels, all_preds

## 📊 Step 11 — Evaluation

In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            inputs = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**inputs)

            logits = outputs.logits
            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels.extend(batch['labels'].numpy())

    return labels, preds

In [ ]:
epochs = 3

total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# To store results for comparison
results = {}

## 🧪 Step 12 — Experiment 1 (Frozen BERT)

In [ ]:
def execute_bert_experiment(exp_name, unfreeze_layers=0):
    print(f"\n{'#'*45}\n# RUNNING: {exp_name.upper()}\n{'#'*45}")

    manager = BERTManager(MODEL_NAME, device)
    model = manager.get_model()

    # Freeze BERT parameters
    for param in model.bert.parameters():
        param.requires_grad = False

    # Selective Unfreezing
    if unfreeze_layers > 0:
        print(f"Unfreezing last {unfreeze_layers} encoder layers...")
        for layer in model.bert.encoder.layer[-unfreeze_layers:]:
            for param in layer.parameters():
                param.requires_grad = True
    else:
        print("Strategy: Feature Extraction (BERT Frozen)")

    # Optimizer with Weight Decay
    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=0.01)

    num_epochs = 3
    total_steps = len(train_loader) * num_epochs

    # Scheduler with Warmup (10% of total steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    for epoch in range(num_epochs):
        avg_loss = manager.train_epoch(model, train_loader, optimizer, scheduler)
        print(f"Epoch {epoch+1}/{num_epochs} | Average Loss: {avg_loss:.4f}")

    y_true, y_pred = manager.validate(model, val_loader)

    results_dict = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred)
    }

    print(f"Final Results for {exp_name}: {results_dict}")
    return results_dict, (y_true, y_pred)

# Run Experiments with optimized logic
results = {}
results['Frozen BERT'], _ = execute_bert_experiment("Frozen BERT", unfreeze_layers=0)
results['Fine-tuned (Last 2)'], _ = execute_bert_experiment("Fine-tuned BERT (Last 2)", unfreeze_layers=2)

## 🔥 Step 13 — Experiment 2 (Fine-Tune Last 2 Layers)

In [ ]:
print("\n--- Running Experiment 2: Fine-Tune Last 2 Layers ---")

# Ensure all BERT parameters are frozen initially (re-freeze from previous experiment)
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze the last 2 layers of BERT's encoder
for layer in model.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# The classifier head parameters are already set to requires_grad=True by default

# Reset optimizer for the new experiment with updated trainable parameters
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

for epoch in range(epochs):
    print(f"Epoch {epoch + 1}/{epochs}")
    train_loss = train(model, train_loader)
    print(f"Training Loss: {train_loss:.4f}")

y_true_exp2, y_pred_exp2 = evaluate(model, val_loader)

accuracy_exp2 = accuracy_score(y_true_exp2, y_pred_exp2)
f1_exp2 = f1_score(y_true_exp2, y_pred_exp2)

results['Fine-tuned BERT'] = {'Accuracy': accuracy_exp2, 'F1 Score': f1_exp2}

print("\nExperiment 2 Results (Validation):")
print(f"Accuracy: {accuracy_exp2:.4f}")
print(f"F1 Score: {f1_exp2:.4f}")
print("Classification Report:\n", classification_report(y_true_exp2, y_pred_exp2))
print("Confusion Matrix:\n", confusion_matrix(y_true_exp2, y_pred_exp2))

## 📈 Step 14 — Metrics

In [ ]:
y_true, y_pred = evaluate(model, val_loader)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1:", f1_score(y_true, y_pred))

print(confusion_matrix(y_true, y_pred))

## 📊 Step 15 — Comparison

| Experiment | Accuracy | F1 Score |
|------------|----------|----------|
| Frozen BERT | ~61%     | ~0.62    |
| Fine-tuned BERT | ~89%     | ~0.89    |


In [ ]:
comparison_df = pd.DataFrame.from_dict(results, orient='index')
print("\n--- Experiment Comparison ---")
display(comparison_df)

## 📌 Analysis & Insights
Frozen BERT works as a feature extractor
Fine-tuning improves performance significantly
Last layers capture task-specific patterns
Trade-off:
Faster training vs better accuracy

## ✅ Summary
Fine-tuning BERT significantly improves performance
Tokenization plays a key role
Partial fine-tuning is optimal for most cases
Transformers outperform traditional NLP models